# Visualización: Gasto de campaña vs. resultado electoral (CORE 2024)

Este notebook toma la base `gasto_vs_resultado_electoral.csv` (gasto de campaña cruzado con el resultado electoral de cada candidatura al Consejo Regional 2024) y construye la visualización atómica de la Entrega 03, usando la librería **Altair**.

**Pregunta que responde esta visualización:** ¿Las candidaturas que declararon más gasto de campaña obtuvieron mejores resultados electorales (más % de votos dentro de su lista)?

## 1. Instalar y cargar librerías
En Colab, `pandas` ya viene instalado. `altair` también viene instalado, pero `vl-convert-python` (que se usa para exportar el gráfico como imagen `.jpg`) hay que instalarlo aparte.

In [ ]:
!pip install -q vl-convert-python

import pandas as pd
import altair as alt


## 2. Cargar la base de datos

Cargamos el CSV directamente desde el repositorio de GitHub del equipo, para no depender de subir el archivo a mano cada vez que se abre el notebook.

In [ ]:
url = "https://raw.githubusercontent.com/amanda-gonzalezz/Repo_Proyecto_Diez_Gonzalez/main/Entrega_03/Diez_Fernanda_01_gasto_vis_01/gasto_vs_resultado_electoral.csv"
df = pd.read_csv(url)
df.shape


In [ ]:
df.head()


## 3. Revisión rápida antes de visualizar

Chequeamos que no haya nulos (la base ya viene limpia de la Entrega 02, pero siempre se revisa antes de graficar) y vemos cómo se distribuye el gasto.

In [ ]:
df.isnull().sum()


In [ ]:
df['gasto_total_declarado'].describe()


## 4. Preparar la variable para graficar

El gasto está en pesos chilenos y los montos son muy grandes, así que lo pasamos a millones de pesos para que el eje del gráfico sea más fácil de leer.

In [ ]:
df['gasto_millones'] = df['gasto_total_declarado'] / 1_000_000
df[['nombre_candidato', 'gasto_total_declarado', 'gasto_millones', 'porcentaje_votos_lista', 'electo']].head()


## 5. Construir la visualización

Un gráfico de dispersión (scatter): cada punto es una candidatura. En el eje X va el gasto declarado (en millones) y en el eje Y el porcentaje de votos que obtuvo dentro de su propia lista. El color distingue si la candidatura resultó electa o no. Se agrega una línea de tendencia (regresión lineal) para ver si existe una correlación.

In [ ]:
dispersión = alt.Chart(df).mark_circle(size=55, opacity=0.55).encode(
    x=alt.X('gasto_millones:Q', title='Gasto de campaña declarado (millones de $)'),
    y=alt.Y('porcentaje_votos_lista:Q', title='% de votos obtenidos dentro de su lista'),
    color=alt.Color(
        'electo:N',
        title='¿Resultó electo/a?',
        scale=alt.Scale(domain=['Sí', 'No'], range=['#2a9d8f', '#e76f51'])
    ),
    tooltip=[
        alt.Tooltip('nombre_candidato:N', title='Candidato/a'),
        alt.Tooltip('partido:N', title='Partido'),
        alt.Tooltip('circunscripcion:N', title='Circunscripción'),
        alt.Tooltip('gasto_millones:Q', title='Gasto (millones $)', format='.1f'),
        alt.Tooltip('posicion_lista:Q', title='Posición en la lista'),
        alt.Tooltip('porcentaje_votos_lista:Q', title='% votos en su lista', format='.1f'),
        alt.Tooltip('electo:N', title='Electo/a')
    ]
).properties(
    width=650,
    height=420,
    title='Gasto de campaña vs. resultado electoral (CORE 2024)'
)

línea_tendencia = dispersión.transform_regression(
    'gasto_millones', 'porcentaje_votos_lista'
).mark_line(color='black', strokeDash=[4, 2])

gráfico_final = (dispersión + línea_tendencia).interactive()
gráfico_final


## 6. Exportar la visualización final

Se exporta en los dos formatos que pide la pauta: `.html` (interactivo, para la webstory) y `.jpg` (imagen estática, para la crónica).

In [ ]:
gráfico_final.save('vis_01.html')
gráfico_final.save('vis_01.jpg', scale_factor=2.0)

print("Listo. Archivos generados: vis_01.html y vis_01.jpg")


## 7. Descargar los archivos desde Colab

Para bajar los dos archivos generados a tu computador (y luego subirlos a la carpeta `visualizacion/` del repositorio):

In [ ]:
from google.colab import files
files.download('vis_01.html')
files.download('vis_01.jpg')
